In [1]:
! pip install timm torchvision

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 5.6 MB/s  0:00:00 eta 0:00:01


In [6]:
import torch 
import torch.nn as nn 
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as T 
from torchvision.datasets import CIFAR10
import timm
from timm.data import Mixup 
from timm.loss import SoftTargetCrossEntropy

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
NUM_CLASSES = 10
BATCH_SIZE = 128 
EPOCHS=10 
LEARNING_RATE = 3e-4
IMAGE_SIZE = 224

Using device: cpu


In [7]:
train_tfms=T.Compose([
    T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    T.RandomHorizontalFlip(),
    T.RandomCrop(IMAGE_SIZE, padding=8),
    T.ToTensor(),
    T.Normalize((0.4914,0.4822,0.4465),(0.2470,0.2435,0.2616))
])
test_tfms=T.Compose([
    T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    T.ToTensor(),
    T.Normalize((0.4914,0.4822,0.4465),(0.2470,0.2435,0.2616))
])


In [9]:
train_ds=CIFAR10(root='./data', train=True, transform=train_tfms, download=True)
test_ds=CIFAR10(root='./data', train=False, transform=test_tfms, download=True)
train_loader=DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
test_loader=DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
len(train_ds), len(test_ds)

Files already downloaded and verified
Files already downloaded and verified


(50000, 10000)

In [10]:
model=timm.create_model('vit_base_patch16_224', pretrained=True, num_classes=NUM_CLASSES)
model.to(device)

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

VisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
    (norm): Identity()
  )
  (pos_drop): Dropout(p=0.0, inplace=False)
  (patch_drop): Identity()
  (norm_pre): Identity()
  (blocks): Sequential(
    (0): Block(
      (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
      (attn): Attention(
        (qkv): Linear(in_features=768, out_features=2304, bias=True)
        (q_norm): Identity()
        (k_norm): Identity()
        (attn_drop): Dropout(p=0.0, inplace=False)
        (norm): Identity()
        (proj): Linear(in_features=768, out_features=768, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (ls1): Identity()
      (drop_path1): Identity()
      (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=768, out_features=3072, bias=True)
        (act): GELU(approximate='none')
        (drop1): Dropout(p=0.0, inplace=False

In [12]:
mixup_fn=Mixup(
    mixup_alpha=0.2,
    cutmix_alpha=1.0,
    prob=0.5,
    switch_prob=0.5,
    mode='batch',
    label_smoothing=0.1,
    num_classes=NUM_CLASSES
)
criterion=SoftTargetCrossEntropy()
optimizer=torch.optim.AdamW(model.parameters(),lr=LEARNING_RATE,weight_decay=0.05)
scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=EPOCHS)

In [15]:
def train_one_epoch(model,loader):
    model.train()
    total_loss,total_correct,total=0.0,0,0

    for x,y in loader:
        x,y=x.to(device,non_blocking=True),y.to(device,non_blocking=True)
        x,y_mix=mixup_fn(x,y)

        optimizer.zero_grad(set_to_none=True)
        logits=model(x)
        loss=criterion(logits,y_mix)
        loss.backward()
        optimizer.step()
        total_loss+=loss.item()*x.size(0)

        preds=logits.argmax(dim=1)
        total_correct+=(preds==y).sum().item()
        total+=x.size(0)
    return total_loss/total, total_correct/total
@ torch.no_grad()
def evaluate(model,loader):
    model.eval()
    total_loss,total_correct,total=0.0,0,0

    for x,y in loader:
        x,y=x.to(device),y.to(device)
        logits=model(x)
        loss=hard_criterion(logits,y)
        total_loss+=loss.item()*x.size(0)
        total_correct+=(logits.argmax(dim=1)==y).sum().item()
        total+=x.size(0)
    return total_loss/total, total_correct/total

In [ ]:
best_acc=0.0
for epoch in range(1,EPOCHS+1):
    train_loss,train_acc=train_one_epoch(model,train_loader)
    test_loss,test_acc=evaluate(model,test_loader)
    scheduler.step()

    if test_acc>best_acc:
        best_acc=test_acc
        torch.save(model.state_dict(),'best_vit_cifar10.pth')
    
    print(f'Epoch [{epoch}/{EPOCHS}] '
          f'Train Loss: {train_loss:.4f} Train Acc: {train_acc:.4f} | '
          f'Test Loss: {test_loss:.4f} Test Acc: {test_acc:.4f} | '
          f'Best Acc: {best_acc:.4f}'
    )